## Pocket constraints with OpenFold3

This notebook co-folds a protein with a small molecule using OpenFold3 and tells the model
which residues line the binding pocket.

Before running this notebook, please ensure you:

1. Are logged in by running `dm login EMAIL` in the terminal
2. Have a Token registered and saved on the file


In [ ]:
!pip install deepmirror

In [ ]:
!dm login <YOUREMAIL>

In [ ]:
import time
import zipfile

import py3Dmol

import deepmirror.api as api
from deepmirror.api import StructureModel

### Define some helper functions

In [ ]:
def cofold(chains: list[dict], user_settings: dict | None = None) -> str:
    response = api.structure_prediction(
        chains, user_settings, model=StructureModel.OPENFOLD3
    )
    task_id = response["batch_job_uuid"]

    while True:
        response = api.get_structure_prediction(task_id)
        status = response["status"]
        if status == "completed":
            break
        if status == "failed":
            raise RuntimeError(f"Structure prediction {task_id} failed")
        print(f"Current status: {status} - Waiting 2 min for completion...")
        time.sleep(120)

    with open(f"result-{task_id}.zip", "wb") as f:
        f.write(api.download_structure_prediction(task_id))

    return task_id


def view_results(task_id: str, highlight_residues: list[int] | None = None):
    with zipfile.ZipFile(f"result-{task_id}.zip", "r") as zf:
        cif_data = zf.read("data.cif").decode("utf-8")
    view = py3Dmol.view(width=500, height=500)
    view.addModel(cif_data, "cif")
    view.setStyle({"cartoon": {"color": "lightgrey"}})
    view.addStyle({"hetflag": True}, {"stick": {}})
    if highlight_residues:
        view.addStyle(
            {"chain": "A", "resi": highlight_residues},
            {"stick": {"colorscheme": "orangeCarbon"}},
        )
    view.zoomTo()
    return view

# Protein + Ligand

The example below is KRAS with its nucleotide GDP. The protein is chain `A` and the ligand is
chain `B`; the chain labels are what the constraint refers to.

In [ ]:
chains = [
    {
        "label": "A",
        "value": "MTEYKLVVVGADGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKEK",
        "type": "protein",
    },
    {
        "label": "B",
        "value": "c1nc2c(n1C3C(C(C(O3)COP(=O)(O)OP(=O)(O)O)O)O)N=C(NC2=O)N",
        "type": "ligand",
    },
]

## Define the pocket constraint

OpenFold3 takes a single `pocket_constraint` object in the settings:

* `ligand_chain_id`: the chain label of the small molecule that sits in the pocket
* `pocket_residues`: a list of `(chain label, residue number)` pairs lining the pocket
* Residue numbers are **1-indexed** positions in the sequence you submit

Other OpenFold3 settings are `num_diffusion_samples` (default 5), `num_model_seeds` (default 1)
and `minimize_predictions` (default `False`).

For KRAS the nucleotide is held by the P-loop (G13, K16, S17), F28, D57, the NKxD motif
(N116, K117, D119) and A146.

In [ ]:
pocket_residues = [13, 16, 17, 28, 57, 116, 117, 119, 146]

user_settings = {
    "num_diffusion_samples": 5,
    "num_model_seeds": 1,
    "pocket_constraint": {
        "ligand_chain_id": "B",
        "pocket_residues": [("A", residue) for residue in pocket_residues],
    },
}

### Initiate Structure Prediction

In [ ]:
task_id = cofold(chains, user_settings)

In [ ]:
view = view_results(task_id, highlight_residues=pocket_residues)
view

### Optional: compare with an unconstrained prediction

Leaving `pocket_constraint` out lets OpenFold3 choose the binding site by itself. Running both
lets you check whether the constraint changed where the ligand was placed.

In [ ]:
task_id_unconstrained = cofold(chains, {"num_diffusion_samples": 5})
view = view_results(task_id_unconstrained, highlight_residues=pocket_residues)
view